# Integrating a bead constrained to a circular guide
Observe three independent diagnostics: radius, tangency and phase. A force-free bead in a circular guide has an exact orbit with which to compare both a constrained step and a simple normalisation. NumPy and Matplotlib are sufficient.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

omega = 1.6                      # s^-1, unit radius in metres
h = .25                          # seconds
steps = 80
assert .1 <= omega <= 3 and .01 <= h and h*omega < .93 and 8 <= steps <= 300

def constrained_step(q, v, dt):
    transverse = dt*v
    # The inward impulse is represented by a multiple of the old unit normal q.
    radial = np.sqrt(1-np.dot(transverse,transverse))
    new_q = radial*q + transverse
    v_half = (new_q-q)/dt
    new_v = v_half - np.dot(v_half,new_q)*new_q
    return new_q, new_v

def normalised_drift(q, v, dt):
    new_q = q+dt*v
    new_q = new_q/np.linalg.norm(new_q)
    new_v = v-np.dot(new_q,v)*new_q
    return new_q,new_v

def trajectory(advance, dt, count):
    q, v = np.array([1.,0.]), np.array([0.,omega])
    positions, velocities = [q.copy()], [v.copy()]
    for _ in range(count):
        q,v = advance(q,v,dt)
        positions.append(q.copy())
        velocities.append(v.copy())
    return np.array(positions),np.array(velocities)

q, v = trajectory(constrained_step,h,steps)
qn, vn = trajectory(normalised_drift,h,steps)
time = np.arange(steps+1)*h
phase = np.unwrap(np.arctan2(q[:,1],q[:,0]))
ang = q[:,0]*v[:,1]-q[:,1]*v[:,0]
print("Largest position/tangency defects:",np.max(abs(np.linalg.norm(q,axis=1)-1)), np.max(abs((q*v).sum(axis=1))))
print(f"Final phase lead {phase[-1]-omega*time[-1]:.6f} rad; angular momentum change {ang[-1]-ang[0]:.3e}")
assert np.max(abs(ang-omega)) < 2e-10 and np.max(abs(phase-np.arange(steps+1)*np.arcsin(h*omega))) < 1e-9

## Read both the shape and the clock
The left panel samples the first part of the trajectory. On the right, unwrapped angles are compared at the same physical times. Halve the step and double the number of steps to retain the final time.

In [ ]:
fig, (path, drift) = plt.subplots(1,2,figsize=(10,4),layout="constrained")
theta = np.linspace(0,2*np.pi,241)
path.plot(np.cos(theta),np.sin(theta),color="grey",lw=1,label="exact guide")
show = min(steps+1, max(4, int(2*np.pi/(omega*h))+1))
path.plot(q[:show,0],q[:show,1],"o-",ms=3,color="#145c71",label="RATTLE nodes")
path.plot(qn[:show,0],qn[:show,1],".--",ms=4,color="#b94626",label="normalised drift")
path.set(aspect="equal",xlabel="x (m)",ylabel="y (m)",title="Both sets of points lie on the guide")
path.legend(fontsize=8)
drift.plot(time, phase-omega*time,color="#145c71",label="RATTLE")
drift.plot(time,np.unwrap(np.arctan2(qn[:,1],qn[:,0]))-omega*time,"--",color="#b94626",label="normalised drift")
drift.set(xlabel="Time (s)",ylabel="Phase error (rad)",title="Constraint satisfaction and timing")
drift.legend()
for ax in (path,drift): ax.grid(alpha=.2)
plt.show()

## Compare a different local solver
An installed OneSimulation environment also lets you integrate the continuous Cartesian acceleration with classical RK4. This alternative implements the centripetal force directly and has no algebraic projection. Compare the end radius with the constrained run at the same step.

In [ ]:
try:
    from onesim.numerics.time import ClassicalRungeKutta4TimeIntegrator
except ImportError:
    print("The constrained and exact references above are complete. OneSimulation comparison is available locally.")
else:
    method = ClassicalRungeKutta4TimeIntegrator()
    def rhs(z, clock):
        pos, vel = np.asarray(z[:2]), np.asarray(z[2:])
        acc = -(vel@vel)/(pos@pos)*pos
        return (*vel, *acc)
    state = (1.,0.,0.,omega)
    for n in range(steps):
        state = method.advance(state,rhs,h,time=n*h)
    print("OneSimulation RK4 end-radius error (m):",np.linalg.norm(state[:2])-1)